# Create a silver layer


### Load the data from the silver layer


#### load crm_cust_info table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import trim, col
from pyspark.sql.types import *

In [0]:
cust_info_df = spark.table("bike_project_catalog.bronze.crm_cust_info")


## Transformation

In [0]:
# trim strings
for field in cust_info_df.schema.fields:
  if isinstance(field.dataType, StringType):
      cust_info_df = cust_info_df.withColumn(field.name, trim(col(field.name)))

In [0]:
# normalization marital status
cust_info_df = cust_info_df.withColumn(
    "cst_marital_status",
    F.when(F.col("cst_marital_status") == "M", "MARRIED")
    .when(F.col("cst_marital_status") == "D", "DIVORCED")
    .when(F.col("cst_marital_status") == "W", "WIDOWED")
    .when(F.col("cst_marital_status") == "S", "SINGLE")
    .otherwise(F.col("cst_marital_status"))
)
# normalization gender
cust_info_df = cust_info_df.withColumn(
    "cst_gndr",
    F.when(F.col("cst_gndr") == "M", "MALE")
    .when(F.col("cst_gndr") == "F", "FEMALE")
    .otherwise(F.col("cst_gndr"))
)

In [0]:
# Rename Columns
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_number",
    "cst_firstname": "first_name",
    "cst_lastname": "last_name",
    "cst_marital_status": "marital_status",
    "cst_gndr": "gender",
    "cst_create_date": "created_date"
}
for old_name, new_name in RENAME_MAP.items():
    cust_info_df = cust_info_df.withColumnRenamed(old_name, new_name)

In [0]:
# Drop rows with null customer_id
cust_info_df = cust_info_df.filter(col("customer_id").isNotNull())

## Write to the Silver layer

In [0]:
cust_info_df.write.mode("overwrite").format("delta").saveAsTable("bike_project_catalog.silver.crm_customers")

In [0]:
%sql
Select count(*) from bike_project_catalog.silver.crm_customers;